Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: Model Armor의 Sanitize Operation Logs를 활용하여 Gemini Enterprise의 모든 프롬프트 본문을 감사용으로 실시간 가로채고, 개인별 토큰 사용량을 집계/추정한다.

## 제미나이 엔터프라이즈 Model Armor 기반 토큰 사용량 모니터링 (Gemini Enterprise Usage by Account with Model Armor )

### 1. 의존성 패키지 설치

빅쿼리 데이터 적재 및 쿼리를 정밀 수행하기 위해 필요한 핵심 클라우드 라이브러리를 설치한다.

In [ ]:
!pip install --quiet google-cloud-bigquery

### 2. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 활성화되어 있는 사용자의 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색하고 전역 변수를 선언한다.

In [ ]:
from google.cloud import bigquery
import google.auth

BIGQUERY_DATASET_ID = "gcp_logs"
BIGQUERY_LOCATION = "asia-northeast3"
DAYS = 7
IAM_ROLE_BIGQUERY = "roles/bigquery.dataEditor"
LOG_SINK_NAME = "model-armor-logs-sink"

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id" # 본인의 실제 GCP 프로젝트 ID로 변경하기 바란다.


### 3. Model Armor 및 로그 싱크 설정 가이드

Model Armor API가 활성화되어 있는지 확인하고, Model Armor의 `SanitizeOperationLogEntry` 로그를 BigQuery 데이터세트로 실시간 우회시키는 라우터 싱크를 생성하고 필요한 빅쿼리 편집자 권한을 부여한다.
터미널에서 아래 gcloud 명령어를 실행하여 인프라를 구성할 수 있다.

```bash
# 1) Model Armor API 활성화
gcloud services enable modelarmor.googleapis.com --project="YOUR_PROJECT_ID"

# 2) 빅쿼리 데이터세트 생성
bq mk --location="asia-northeast3" --dataset "YOUR_PROJECT_ID:gcp_logs"

# 3) 빅쿼리 목적지 로그 싱크 생성
gcloud logging sinks create "model-armor-logs-sink" \
  "bigquery.googleapis.com/projects/YOUR_PROJECT_ID/datasets/gcp_logs" \
  --log-filter='jsonPayload.@type="type.googleapis.com/google.cloud.modelarmor.logging.v1.SanitizeOperationLogEntry"' \
  --project="YOUR_PROJECT_ID"

# 4) 생성된 싱크의 서비스 계정 권한 위임
LOG_SINK_SA=$(gcloud logging sinks describe "model-armor-logs-sink" --project="YOUR_PROJECT_ID" --format="value(writerIdentity)")
gcloud projects add-iam-policy-binding "YOUR_PROJECT_ID" \
  --member="$LOG_SINK_SA" \
  --role="roles/bigquery.dataEditor"
```

### 4. 빅쿼리에서 SQL로 사용자별 실시간 토큰 사용량 및 점유 추정 조회 실행

빅쿼리 목적지 테이블에 실시간으로 우회 적재되는 Model Armor 감사 로그를 SQL로 분석하여, 최근 7일 동안 어떤 계정이 Model Armor를 통해 제미나이 엔터프라이즈를 얼마나 사용했는지 추정 토큰 수량 순위 통계를 조회한다.

**토큰 추정 공식:**
* 영문 4자당 1토큰(글자 수 * 1.2 가중치), 한글 1자당 1.5토큰(글자 수 * 1.5 가중치) 수준을 적용하여 가중 계산을 수행한다.
* Model Armor 로그에서 `jsonPayload.userPrompt.content`와 `jsonPayload.modelResponse.content`를 파싱하여 활용한다.

**빅쿼리 비용 최적화 설계:**
* **과금 차단용 파티션 필터**: 대량의 데이터 스캔으로 인한 불필요한 빅쿼리 분석 비용 폭증을 철저하게 방지하기 위해, `_TABLE_SUFFIX` 파티션 필터 조건절을 WHERE 조건 내에 명시하여 최근 7일 동안 생성된 와일드카드 감사 로그 테이블만 강제 제한 스캔하도록 설계한다.

In [ ]:
try:
  bq_client = bigquery.Client(project=project_id)
  query = f"""
    SELECT 
      SUM(
        CAST(
          ROUND(
            CHARACTER_LENGTH(JSON_VALUE(jsonPayload.userPrompt.content)) * 1.2
          ) AS INT64
        )
      ) AS estimated_prompt_tokens,
      
      SUM(
        CAST(
          ROUND(
            CHARACTER_LENGTH(JSON_VALUE(jsonPayload.modelResponse.content)) * 1.5
          ) AS INT64
        )
      ) AS estimated_response_tokens,
      
      SUM(
        CAST(
          ROUND(
            (CHARACTER_LENGTH(JSON_VALUE(jsonPayload.userPrompt.content)) * 1.2) +
            (CHARACTER_LENGTH(JSON_VALUE(jsonPayload.modelResponse.content)) * 1.5)
          ) AS INT64
        )
      ) AS estimated_total_tokens,
      
      COUNT(1) AS inspection_count,
      
      COALESCE(
        JSON_VALUE(jsonPayload.metadata.client_correlation_id),
        JSON_VALUE(labels["modelarmor.googleapis.com/client_name"]),
        'unknown_principal'
      ) AS principal_identity

    FROM `{project_id}.{BIGQUERY_DATASET_ID}.modelarmor_googleapis_com_sanitize_operations_*`
    WHERE _TABLE_SUFFIX >= FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL {DAYS} DAY))
    GROUP BY principal_identity
    ORDER BY estimated_total_tokens DESC;
  """
  
  query_job = bq_client.query(query)
  results = query_job.result()
  
  print("\n=== [사용자별 제미나이 엔터프라이즈 Model Armor 기반 토큰 사용량 통계] ===")
  for row in results:
    print(f"사용자 식별자: {row.principal_identity} | "
          f"검사 횟수: {row.inspection_count}회 | "
          f"추정 프롬프트 토큰: {row.estimated_prompt_tokens} | "
          f"추정 응답 토큰: {row.estimated_response_tokens} | "
          f"총 추정 토큰: {row.estimated_total_tokens}")
except Exception as e:
  print(f"[안내] 빅쿼리 데이터 조회 중 예외가 발생했으나 계속 진행한다 (테이블이 아직 생성되지 않았을 수 있음): {e}")
